In [2]:
import pandas as pd

master = pd.read_csv('results_ohne_UNKNOWN/master_predictions.csv')
NUMERIC = {'read_speed_mb_s','write_speed_mb_s','height_mm','width_mm'}
CONFIGS = ['llm_prediction','rag_minilm_prediction','rag_minilm_reranker_prediction',
           'rag_bge_reranker_prediction','rag_te_reranker_prediction']

def exact_numeric(predicted, ground_truth, attribute):
    if attribute not in NUMERIC: return None  # skip text attrs
    if not predicted or str(predicted).lower() in {'unknown','nan','none',''}: return False
    try:
        p = float(str(predicted).replace(',','').strip())
        g = float(str(ground_truth).replace(',','').strip())
        return p == g
    except: return False

numeric = master[master['attribute'].isin(NUMERIC)].copy()

print(f'{"Config":<35} {"±10% acc":>10} {"Exact acc":>10} {"Diff":>8}')
print('-' * 65)
for col in CONFIGS:
    rows = numeric.dropna(subset=[col])
    tol   = rows.apply(lambda r: abs(float(str(r[col]).replace(',','') or 0) - 
                       float(str(r['ground_truth']).replace(',','') or 0)) / 
                       abs(float(str(r['ground_truth']).replace(',','') or 1)) <= 0.10 
                       if str(r[col]).lower() not in {'unknown','nan','none',''} else False, axis=1).mean()
    exact = rows.apply(lambda r: exact_numeric(r[col], r['ground_truth'], r['attribute']), axis=1).mean()
    label = col.replace('_prediction','')
    print(f'{label:<35} {tol*100:>9.1f}% {exact*100:>9.1f}% {(exact-tol)*100:>+7.1f}%')

Config                                ±10% acc  Exact acc     Diff
-----------------------------------------------------------------


ValueError: could not convert string to float: 'None (no sequential write speed mentioned)'

In [3]:
import pandas as pd

master = pd.read_csv('results_ohne_UNKNOWN/master_predictions.csv')
NUMERIC = {'read_speed_mb_s','write_speed_mb_s','height_mm','width_mm'}
CONFIGS = ['llm_prediction','rag_minilm_prediction','rag_minilm_reranker_prediction',
           'rag_bge_reranker_prediction','rag_te_reranker_prediction']

numeric = master[master['attribute'].isin(NUMERIC)].copy()

def is_tol(pred, gt):
    try:
        p = float(str(pred).replace(',','').strip())
        g = float(str(gt).replace(',','').strip())
        return abs(p-g)/abs(g) <= 0.10 if g != 0 else p == 0
    except: return False

def is_exact(pred, gt):
    try:
        p = float(str(pred).replace(',','').strip())
        g = float(str(gt).replace(',','').strip())
        return p == g
    except: return False

def is_unknown(pred):
    return str(pred).strip().lower() in {'unknown','nan','none',''}

# Show only rows where tol=correct but exact=wrong (the interesting cases)
print(f'{"Row":<5} {"Attr":<20} {"GT":<12} {"Predicted":<12} {"Tol":>5} {"Exact":>6} {"Diff%":>7}  Config')
print('='*90)

for col in CONFIGS:
    label = col.replace('_prediction','')
    rows = numeric.dropna(subset=[col])
    for _, r in rows.iterrows():
        pred = r[col]
        gt   = r['ground_truth']
        if is_unknown(pred): continue
        tol   = is_tol(pred, gt)
        exact = is_exact(pred, gt)
        if tol and not exact:  # tol passes but exact fails — the interesting gap
            try:
                p = float(str(pred).replace(',','').strip())
                g = float(str(gt).replace(',','').strip())
                diff = (p-g)/g*100
            except: diff = 0
            print(f'{int(r["df1_idx"]):<5} {r["attribute"]:<20} {str(gt):<12} {str(pred):<12} {"✓":>5} {"✗":>6} {diff:>+6.1f}%  {label}')

Row   Attr                 GT           Predicted      Tol  Exact   Diff%  Config
77    read_speed_mb_s      555.0        520              ✓      ✗   -6.3%  llm
77    write_speed_mb_s     520.0        540              ✓      ✗   +3.8%  llm
135   write_speed_mb_s     540.0        520              ✓      ✗   -3.7%  llm
152   write_speed_mb_s     530.0        520              ✓      ✗   -1.9%  llm
18    write_speed_mb_s     520.0        550              ✓      ✗   +5.8%  rag_minilm
28    height_mm            42.0         46               ✓      ✗   +9.5%  rag_minilm
59    width_mm             690.0        710              ✓      ✗   +2.9%  rag_minilm
60    read_speed_mb_s      560.0        550              ✓      ✗   -1.8%  rag_minilm
90    width_mm             119.3        127              ✓      ✗   +6.5%  rag_minilm
131   write_speed_mb_s     430.0        450              ✓      ✗   +4.7%  rag_minilm
152   read_speed_mb_s      560.0        550.0            ✓      ✗   -1.8%  rag_minilm


In [4]:
import pandas as pd

df1 = pd.read_json('normalized_products/dataset_1_normalized.json')
df2 = pd.read_json('normalized_products/dataset_2_normalized.json')
df3 = pd.read_json('normalized_products/dataset_3_normalized.json')
df4 = pd.read_json('normalized_products/dataset_4_normalized.json')
kb  = pd.concat([df2, df3, df4], ignore_index=True)

master   = pd.read_csv('results_ohne_UNKNOWN/master_predictions.csv')
eval_df  = pd.read_csv('eval_set.csv')
NUMERIC  = {'read_speed_mb_s','write_speed_mb_s','height_mm','width_mm'}

numeric_tasks = eval_df[eval_df['attribute'].isin(NUMERIC)].copy()

print(f'{"Row":<5} {"Attr":<20} {"GT (used)":<12} {"All KB values for this cluster":<50} {"Consistent?"}')
print('='*105)

consistent = 0
inconsistent = 0

for _, task in numeric_tasks.iterrows():
    idx        = task['df1_idx']
    attr       = task['attribute']
    gt         = task['ground_truth']
    cluster_id = df1.loc[idx, 'cluster_id']

    # Get all non-null values for this attr from KB
    kb_matches = kb[kb['cluster_id'] == cluster_id]
    all_vals = []
    for _, row in kb_matches.iterrows():
        val = row.get(attr)
        if pd.notna(val) and str(val).strip().lower() not in {'', 'none', 'nan'}:
            try:
                all_vals.append(float(str(val).replace(',','').strip()))
            except:
                all_vals.append(str(val).strip())

    if len(all_vals) == 0:
        continue

    # Check if all values are the same
    unique_vals = list(set(str(v) for v in all_vals))
    same = len(unique_vals) == 1
    consistent   += int(same)
    inconsistent += int(not same)

    status = '✓ same' if same else '✗ DIFFERENT'
    print(f'{int(idx):<5} {attr:<20} {str(gt):<12} {str(all_vals):<50} {status}')

print(f'\nConsistent (all KB values same):     {consistent}')
print(f'Inconsistent (KB values differ):     {inconsistent}')
print(f'Inconsistency rate:                  {inconsistent/(consistent+inconsistent)*100:.1f}%')

Row   Attr                 GT (used)    All KB values for this cluster                     Consistent?
2     read_speed_mb_s      3480.0       [3480.0, 3480.0]                                   ✓ same
2     write_speed_mb_s     3000.0       [3000.0, 3000.0]                                   ✓ same
3     read_speed_mb_s      226.0        [226.0]                                            ✓ same
4     width_mm             433.0        [433.0]                                            ✓ same
9     height_mm            46.0         [46.0]                                             ✓ same
9     width_mm             127.0        [127.0]                                            ✓ same
18    read_speed_mb_s      550.0        [550.0, 550.0]                                     ✓ same
18    write_speed_mb_s     520.0        [520.0, 520.0]                                     ✓ same
18    height_mm            7.0          [7.0]                                              ✓ same
20    read_spee